In [169]:
import numpy as np
import pandas as pd

In [170]:
df=pd.read_csv("imdb_raw.csv")
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   title         1000 non-null   str    
 1   director      1000 non-null   str    
 2   release_year  1000 non-null   str    
 3   runtime       1000 non-null   str    
 4   genre         1000 non-null   str    
 5   rating        1000 non-null   float64
 6   metascore     1000 non-null   int64  
 7   gross         1000 non-null   str    
dtypes: float64(1), int64(1), str(6)
memory usage: 62.6 KB


for merging purpose

In [171]:
budget_data = pd.DataFrame({
    'title': df['title'].unique(),
    'budget': np.random.randint(5, 200, size=len(df['title'].unique()))
})

cleaning data :- release year had bracket removed it and type conversion done to int

In [172]:
df['release_year'] = df['release_year'].str.extract(r'(\d{4})').astype(int)

remove min for analysis

In [173]:
df['runtime']=df['runtime'].str.replace(' min','').astype(int)

Remove signs and characters and zero value handling

In [174]:
df['gross']=df['gross'].str.replace('$','').str.replace('M','')
df['gross']=pd.to_numeric(df['gross'])

In [175]:
df['gross']=df['gross'].replace(0,np.nan)

simplyfing genre


In [176]:
df['genre']=df['genre'].str.split(',').str[0]

cleaning is done

NOW MERGE

In [177]:
df_final = pd.merge(df, budget_data, on='title', how='left')

filling missing values with median for budget if any

In [178]:
df_final['budget']=df_final['budget'].fillna(df_final['budget']).median()

Feature engineering

In [179]:
df_final['Profit_Millions']=df['gross']-df_final['budget']
df_final

,title,director,release_year,runtime,genre,rating,metascore,gross,budget,Profit_Millions
0,The Shawshank Redemption,Frank Darabont,1994,142,Drama,9.3,82,28.34,98.0,-69.66
1,The Godfather,Francis Ford Coppola,1972,175,Crime,9.2,100,134.97,98.0,36.97
2,The Dark Knight,Christopher Nolan,2008,152,Action,9.0,84,534.86,98.0,436.86
3,Schindler's List,Steven Spielberg,1993,195,Biography,9.0,95,96.90,98.0,-1.10
4,12 Angry Men,Sidney Lumet,1957,96,Crime,9.0,97,4.36,98.0,-93.64
...,...,...,...,...,...,...,...,...,...,...
995,A Very Long Engagement,Jean-Pierre Jeunet,2004,133,Drama,7.6,76,6.17,98.0,-91.83
996,Shine,Scott Hicks,1996,105,Biography,7.6,87,35.81,98.0,-62.19
997,Philomena,Stephen Frears,2013,98,Biography,7.6,77,37.71,98.0,-60.29
998,The Invisible Man,James Whale,1933,71,Horror,7.6,87,NaN,98.0,NaN


In [180]:
df_final['ROI']=df_final['Profit_Millions']/df_final['budget']
df_final

,title,director,release_year,runtime,genre,rating,metascore,gross,budget,Profit_Millions,ROI
0,The Shawshank Redemption,Frank Darabont,1994,142,Drama,9.3,82,28.34,98.0,-69.66,-0.710816
1,The Godfather,Francis Ford Coppola,1972,175,Crime,9.2,100,134.97,98.0,36.97,0.377245
2,The Dark Knight,Christopher Nolan,2008,152,Action,9.0,84,534.86,98.0,436.86,4.457755
3,Schindler's List,Steven Spielberg,1993,195,Biography,9.0,95,96.90,98.0,-1.10,-0.011224
4,12 Angry Men,Sidney Lumet,1957,96,Crime,9.0,97,4.36,98.0,-93.64,-0.955510
...,...,...,...,...,...,...,...,...,...,...,...
995,A Very Long Engagement,Jean-Pierre Jeunet,2004,133,Drama,7.6,76,6.17,98.0,-91.83,-0.937041
996,Shine,Scott Hicks,1996,105,Biography,7.6,87,35.81,98.0,-62.19,-0.634592
997,Philomena,Stephen Frears,2013,98,Biography,7.6,77,37.71,98.0,-60.29,-0.615204
998,The Invisible Man,James Whale,1933,71,Horror,7.6,87,NaN,98.0,NaN,NaN


grouping to read insight together on basis of genre

In [182]:
genre_based_analysis = df_final.groupby('genre')[['Profit_Millions','ROI','rating']].mean().sort_values(by=['rating'],ascending=False)
genre_based_analysis

,Profit_Millions,ROI,rating
genre,,,
Western,-79.440000,-0.810612,8.150000
Mystery,-61.461429,-0.627157,8.100000
Crime,-64.609891,-0.659285,8.044037
Adventure,1.941569,0.019812,7.988889
Action,58.294490,0.594842,7.985106
Biography,-38.844211,-0.396369,7.968966
Drama,-61.899607,-0.631629,7.966552
Film-Noir,-97.550000,-0.995408,7.950000
Animation,26.272794,0.268090,7.942857
